# 04 · Eleição do vencedor e promoção controlada

**Etapa 1 — vencedor do ciclo:** maior média de AUC-PR na validação cruzada (notebook 03). Empate: menor desvio-padrão.

**Etapa 2 — vencedor × vigente:** os dois são avaliados na **mesma validação do ciclo** (dados que nenhum dos dois viu no treino).

| Situação | Decisão |
|---|---|
| Candidato não supera o modelo aleatório (AUC-PR ≤ prevalência) | REJEITADO, vigente mantido |
| Não existe vigente | PROMOVIDO |
| Candidato ≥ vigente + 0,005 | PROMOVIDO |
| Caso contrário | MANTIDO |

O teste (15% reservado) só é usado no fim, para reportar o desempenho do modelo que ficou vigente. Qualquer erro neste notebook interrompe o workflow **antes** de o novo estado ser salvo, então nunca há promoção indevida.

In [1]:
import os, sys, warnings
from pathlib import Path
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
warnings.filterwarnings("ignore", category=UserWarning)
# Localiza a raiz do repositório e torna o pacote src importável
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
os.chdir(RAIZ); sys.path.insert(0, str(RAIZ))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from src.utils import carregar_config, configurar_mlflow, ler_contexto, atualizar_contexto, tags_ciclo, caminho, salvar_json
cfg = carregar_config()
pd.set_option("display.width", 160)

In [2]:
import joblib
from src.preprocessamento import separar_xy
from src.modelagem import calcular_metricas
from src.promocao import eleger_vencedor, decidir_promocao, carregar_vigente, promover, registrar_decisao, ler_manifesto
from src.utils import resumo_github

ctx = ler_contexto(cfg)
metrica, direcao = cfg["avaliacao"]["metrica_principal"], cfg["avaliacao"]["direcao"]
pct = cfg["negocio"]["percentual_contato"]
proc = caminho(cfg["dados"]["processed_dir"])
validacao = pd.read_csv(proc / "validacao.csv"); teste = pd.read_csv(proc / "teste.csv")
X_va, y_va = separar_xy(validacao, cfg); X_te, y_te = separar_xy(teste, cfg)
cv = pd.read_csv(caminho(cfg["dados"]["state_dir"]) / "cv_ciclo.csv")
print(ctx["cycle_id"], ctx["data_version"])

c006-20261121-1758 v006-c063ab96


## 1. Vencedor do ciclo (A × B)

In [3]:
vencedor, ranking = eleger_vencedor(cv, metrica, direcao)
display(ranking.round(4))
cand = ctx["candidatos"][vencedor]
pipe_cand = joblib.load(caminho(cfg["dados"]["state_dir"]) / "candidatos" / f"{vencedor}.joblib")
print(f"Vencedor do ciclo: Modelo {vencedor} ({cand['algoritmo']})")

,modelo,algoritmo,average_precision_media,average_precision_desvio
0,B,random_forest,0.4016,0.0225
1,A,regressao_logistica,0.3847,0.0205


Vencedor do ciclo: Modelo B (random_forest)


## 2. Vencedor × modelo vigente (mesma validação)

In [4]:
vigente, manif_ant = carregar_vigente()   # vem do armazenamento persistente (repositório), não do runner
m_cand = calcular_metricas(y_va, pipe_cand.predict_proba(X_va)[:, 1], pct)
m_vig = calcular_metricas(y_va, vigente.predict_proba(X_va)[:, 1], pct) if vigente is not None else None

# Verificações de sanidade: falhas aqui interrompem o fluxo antes da promoção
assert len(ctx["features_eleitas"]) > 0, "Conjunto de features vazio"
assert np.isfinite(m_cand[metrica]), "Métrica do candidato inválida"
assert list(pipe_cand.named_steps["selecao"].features) == ctx["features_eleitas"], "Candidato não usa as features eleitas do ciclo"

piso = m_cand["prevalencia"]
decisao = decidir_promocao(m_cand[metrica], None if m_vig is None else m_vig[metrica], piso, cfg)

comp = pd.DataFrame({f"candidato ({cand['algoritmo']}, {ctx['cycle_id']})": m_cand,
                     **({f"vigente ({manif_ant['algoritmo']}, {manif_ant['cycle_id']})": m_vig} if m_vig else {})}).round(4)
display(comp)
display(Markdown(f"### Decisão: **{decisao['decisao']}**  \n{decisao['motivo']}"))

,"candidato (random_forest, c006-20261121-1758)","vigente (random_forest, c005-20261120-1906)"
average_precision,0.4154,0.3901
roc_auc,0.8008,0.7987
f1,0.4155,0.4060
precision,0.3081,0.2949
recall,0.6375,0.6512
prevalencia,0.0982,0.0982
conversao_top,0.3105,0.3114
lift_top,3.1626,3.1712
captura_top,0.6323,0.6340


### Decisão: **PROMOVIDO**  
Candidato (0.4154) supera o vigente (0.3901) em pelo menos 0.005 na validação do ciclo.

## 3. Aplicação da decisão e manifesto

In [5]:
if decisao["decisao"] == "PROMOVIDO":
    manifesto = promover(pipe_cand, {
        "cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "modelo": vencedor,
        "algoritmo": cand["algoritmo"], "run_id": cand["run_id"], "features": ctx["features_eleitas"],
        "metrica_principal": metrica, "valor_validacao": m_cand[metrica],
        "cv_media": cand[f"cv_{metrica}_media"], "decisao": decisao,
        "drift_alerta_no_ciclo": ctx.get("drift_alerta"), "features_com_drift": ctx.get("features_com_drift"),
        "gatilho": ctx["execucao"]["gatilho"]})
    vigente = pipe_cand
else:
    manifesto = ler_manifesto()
    if manifesto is None:
        raise RuntimeError("Nenhum modelo vigente e o candidato foi rejeitado: pipeline interrompido.")
print(json.dumps(manifesto, indent=2, ensure_ascii=False))

{
  "cycle_id": "c006-20261121-1758",
  "data_version": "v006-c063ab96",
  "modelo": "B",
  "algoritmo": "random_forest",
  "run_id": "cbb0009f675e44da9da55613162a527d",
  "features": [
    "euribor3m",
    "cons.price.idx",
    "emp.var.rate",
    "poutcome",
    "cons.conf.idx",
    "month",
    "contact",
    "nr.employed"
  ],
  "metrica_principal": "average_precision",
  "valor_validacao": 0.4153782739308943,
  "cv_media": 0.4016477921000542,
  "decisao": {
    "decisao": "PROMOVIDO",
    "motivo": "Candidato (0.4154) supera o vigente (0.3901) em pelo menos 0.005 na validação do ciclo."
  },
  "drift_alerta_no_ciclo": true,
  "features_com_drift": [
    "nr.employed",
    "euribor3m",
    "cons.conf.idx",
    "emp.var.rate",
    "cons.price.idx",
    "month",
    "poutcome",
    "previous",
    "job",
    "faixa_etaria",
    "default",
    "age"
  ],
  "gatilho": "schedule",
  "arquivo_modelo": "models/historico/c006-20261121-1758_random_forest.joblib",
  "promovido_em": "2026-11-

## 4. Avaliação final do modelo vigente no teste reservado

In [6]:
m_teste = calcular_metricas(y_te, vigente.predict_proba(X_te)[:, 1], pct)
display(pd.Series(m_teste, name="teste").round(4).to_frame())

mlflow = configurar_mlflow(cfg)
with mlflow.start_run(run_name=f"{ctx['cycle_id']}_04_decisao", tags={**tags_ciclo(ctx, "04_promocao"), "decisao": decisao["decisao"]}) as run:
    mlflow.log_params({"vencedor_ciclo": vencedor, "algoritmo_vencedor": cand["algoritmo"], "run_id_vencedor": cand["run_id"],
                       "decisao": decisao["decisao"], "motivo": decisao["motivo"][:250],
                       "limiar_melhoria": cfg["promocao"]["limiar_melhoria"],
                       "vigente_cycle_id": manifesto["cycle_id"], "vigente_run_id": manifesto["run_id"]})
    mlflow.log_metrics({f"candidato_val_{metrica}": m_cand[metrica], f"piso_aleatorio_{metrica}": piso,
                        **({f"vigente_anterior_val_{metrica}": m_vig[metrica]} if m_vig else {}),
                        **{f"vigente_teste_{k}": v for k, v in m_teste.items()}})
    mlflow.log_dict(manifesto, "modelo_vigente.json")
    mlflow.log_table(ranking, "ranking_vencedor.json")

linha = {"cycle_id": ctx["cycle_id"], "data_version": ctx["data_version"], "gatilho": ctx["execucao"]["gatilho"],
         "lote": ctx["lote"].get("lote"), "lote_novo": ctx["lote"]["novo"], "linhas_lote": ctx["lote"]["linhas"],
         "features_eleitas": "|".join(ctx["features_eleitas"]), "n_features": len(ctx["features_eleitas"]),
         "cv_A": ctx["candidatos"]["A"][f"cv_{metrica}_media"], "cv_B": ctx["candidatos"]["B"][f"cv_{metrica}_media"],
         "vencedor": f"{vencedor}-{cand['algoritmo']}", f"candidato_val_{metrica}": m_cand[metrica],
         f"vigente_anterior_val_{metrica}": None if m_vig is None else m_vig[metrica],
         "decisao": decisao["decisao"], "vigente_apos": f"{manifesto['cycle_id']}-{manifesto['algoritmo']}",
         f"vigente_teste_{metrica}": m_teste[metrica], "vigente_teste_lift_top": m_teste["lift_top"],
         "drift_alerta": ctx.get("drift_alerta"), "psi_max": ctx.get("psi_max"), "run_decisao": run.info.run_id}
historico = registrar_decisao(linha)
display(historico.tail(5))

resumo_github(f"### Decisão: **{decisao['decisao']}**\n\n{decisao['motivo']}\n\n"
              f"| | valor |\n|---|---|\n| vencedor do ciclo | {vencedor} ({cand['algoritmo']}) |\n"
              f"| features | {', '.join(ctx['features_eleitas'])} |\n| vigente | {manifesto['cycle_id']} ({manifesto['algoritmo']}) |\n"
              f"| {metrica} no teste (vigente) | {m_teste[metrica]:.4f} |\n")
atualizar_contexto(cfg, decisao=decisao, run_decisao=run.info.run_id)

,teste
average_precision,0.4047
roc_auc,0.7652
f1,0.4155
precision,0.3197
recall,0.5931
prevalencia,0.1038
conversao_top,0.3122
lift_top,3.0071
captura_top,0.6013


,cycle_id,data_version,gatilho,lote,lote_novo,linhas_lote,features_eleitas,n_features,cv_A,cv_B,vencedor,candidato_val_average_precision,vigente_anterior_val_average_precision,decisao,vigente_apos,vigente_teste_average_precision,vigente_teste_lift_top,drift_alerta,psi_max,run_decisao
1,c002-20261117-1951,v002-db4f8bd6,schedule,1,True,2750,euribor3m|month|emp.var.rate|nr.employed|cons....,8,0.191662,0.205791,B-random_forest,0.204310,0.106718,PROMOVIDO,c002-20261117-1951-random_forest,0.183092,2.093245,True,16.4750,559ce72fad6c46a6862a5f5f2141eb78
2,c003-20261118-1924,v003-31eeff6c,schedule,2,True,6687,euribor3m|emp.var.rate|contact|month|cons.pric...,8,0.234072,0.234779,B-random_forest,0.255523,0.222392,PROMOVIDO,c003-20261118-1924-random_forest,0.228976,2.352408,True,10.1202,776a93e80ee54b5bbdaa3585bd9b94b8
3,c004-20261119-1929,v004-ce2b8d77,schedule,3,True,1484,euribor3m|nr.employed|cons.price.idx|emp.var.r...,8,0.298226,0.301461,B-random_forest,0.320871,0.281793,PROMOVIDO,c004-20261119-1929-random_forest,0.292896,2.636831,True,11.7757,062bd66469a14c0cbe969b39c33fe11a
4,c005-20261120-1906,v005-0c33f7d6,schedule,4,True,793,euribor3m|cons.price.idx|emp.var.rate|poutcome...,8,0.348007,0.361516,B-random_forest,0.362801,0.347027,PROMOVIDO,c005-20261120-1906-random_forest,0.359554,2.918149,True,10.2597,49593514c9da41e9bc194d4c4f20b56d
5,c006-20261121-1758,v006-c063ab96,schedule,5,True,615,euribor3m|cons.price.idx|emp.var.rate|poutcome...,8,0.384731,0.401648,B-random_forest,0.415378,0.390053,PROMOVIDO,c006-20261121-1758-random_forest,0.404742,3.007104,True,9.7881,2cc180490c5d4a33b896c354b7f1a560


{'cycle_id': 'c006-20261121-1758',
 'data_version': 'v006-c063ab96',
 'execucao': {'gatilho': 'schedule',
  'github_run_id': '44932827856',
  'github_run_url': 'https://github.com/weillerjulia/n3-mlops-bank-marketing/actions/runs/44932827856',
  'commit': 'a0db65c',
  'iniciado_em': '2026-11-21T17:58:15+00:00'},
 'lote': {'novo': True,
  'lote': 5,
  'arquivo': 'data/raw/lote_005.csv',
  'periodos': ['2010-04', '2010-05', '2010-06'],
  'linhas': 615,
  'sha256': '979101a920a1',
  'ingerido_em': '2026-11-21T17:58:15+00:00'},
 'drift_alerta': True,
 'features_com_drift': ['nr.employed',
  'euribor3m',
  'cons.conf.idx',
  'emp.var.rate',
  'cons.price.idx',
  'month',
  'poutcome',
  'previous',
  'job',
  'faixa_etaria',
  'default',
  'age'],
 'psi_max': 9.7881,
 'run_etl': 'ad7ee1a637e34534ba0c3ce8815d9953',
 'features_eleitas': ['euribor3m',
  'cons.price.idx',
  'emp.var.rate',
  'poutcome',
  'cons.conf.idx',
  'month',
  'contact',
  'nr.employed'],
 'run_votacao': 'cd9c988888ac4c